No,３

In [3]:
import os
import re
from pathlib import Path


# --- 共通の復元基盤 undo_utils.py を読み込む ---
# ※ このセルの import を並べ替えても壊れないように、undo_utils だけは
#    import 文ではなく importlib 経由で読み込んでいます。
import importlib
import sys
from pathlib import Path


def _locate_undo_utils():
    """undo_utils.py があるフォルダを探す（VS Code の作業ディレクトリ設定に依存しない）。"""
    candidates = []
    # 1) VS Code がノートブック自身のパスを教えてくれる場合
    nb_file = globals().get("__vsc_ipynb_file__")
    if nb_file:
        candidates.append(Path(nb_file).parent)
    # 2) 作業ディレクトリと、その中／親の「コードフォルダ」
    cwd = Path.cwd()
    candidates += [cwd, cwd / "コードフォルダ", cwd.parent, cwd.parent / "コードフォルダ"]

    for c in candidates:
        if (c / "undo_utils.py").is_file():
            return c.resolve()

    raise FileNotFoundError(
        "undo_utils.py が見つかりません。\n"
        "このノートブックと同じ「コードフォルダ」内に undo_utils.py があるか確認してください。\n"
        f"探した場所: {[str(c) for c in candidates]}"
    )


_uu_dir = str(_locate_undo_utils())
if _uu_dir not in sys.path:
    sys.path.insert(0, _uu_dir)

uu = importlib.import_module("undo_utils")
importlib.reload(uu)  # undo_utils.py を編集した場合も反映されるようにする

<module 'undo_utils' from 'C:\\Users\\0uh2j\\Desktop\\vscodeで\\ファイル整理２\\コードフォルダ\\undo_utils.py'>

末端フォルダ内のファイル数を調整：設定したファイル数を超える場合のみ適用する。削除するファイルは昇順。

> **⚠️ 変更点**: 以前はファイルを完全削除（`os.remove`）していましたが、
> 現在は削除せずにメインフォルダ直下の `_trash` フォルダへ**退避**します。
> 末尾の復元セルを実行すれば元の場所へ戻せます。
> 問題ないことを確認できたら、`_trash` フォルダを手動で削除してください。

NR用

In [2]:
# === NR用: 末端フォルダごとにファイル数を KEEP_COUNT 件に調整 ===
# 超過分は削除せず _trash へ退避します（復元セルで元に戻せます）。

KEEP_COUNT = 10  # 各フォルダに残すファイル数
STEP_NAME = "03_ファイル数調整 (NR用)"


def main():
    base_dir = uu.select_folder("大元のフォルダを選択してください")
    if base_dir is None:
        return

    print("-" * 30)

    # ファイル名のルール: 特定の数字 + "$" + 連番 + 拡張子
    pattern = re.compile(r"^([^$]+)\$(\d+)(.*)$")

    trashed_total = 0

    # uu.walk は _undo / _trash 以下に降りない
    #   → 退避済みのファイルを再処理してしまう事故を防ぐ
    with uu.UndoJournal(base_dir, STEP_NAME) as j:
        for dirpath, dirnames, filenames in uu.walk(base_dir):
            target_files = []

            for filename in filenames:
                match = pattern.match(filename)
                if match:
                    seq_num = int(match.group(2))
                    target_files.append((filename, seq_num))
                # ルールに合致しないファイルはスキップ（安全対策）

            if len(target_files) > KEEP_COUNT:
                # 連番の小さい順にソートし、古い方から退避
                target_files.sort(key=lambda x: x[1])
                trash_count = len(target_files) - KEEP_COUNT

                for i in range(trash_count):
                    file_path = dirpath / target_files[i][0]
                    try:
                        j.trash(file_path)
                        print(f"退避しました: {file_path}")
                        trashed_total += 1
                    except Exception as e:
                        print(f"退避エラー ({file_path}): {e}")

    print("-" * 30)
    print(f"処理が完了しました。合計 {trashed_total} 個のファイルを _trash へ退避しました。")


main()

選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/202603-4 - 本実験圧力データ/202606・08-実験データ再々/202606・08-直接式樹脂圧力本実験データ再々/1.0～2.0mm/csv
------------------------------
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\csv\1.0mm\190℃\010\010$0.csv
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\csv\1.0mm\190℃\010\010$1.csv
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\csv\1.0mm\190℃\010\010$2.csv
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\csv\1.0mm\190℃\020\020$0.csv
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\csv\1.0mm\190℃\020\020$1.csv
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力

Futaba用

In [4]:
# === Futaba用: 日付・時刻を無視してグループ化し、ファイル数を調整 ===
# 超過分は削除せず _trash へ退避します（復元セルで元に戻せます）。

KEEP_COUNT = 10  # 各グループに残すファイル数
STEP_NAME = "03_ファイル数調整 (Futaba用)"


def main():
    base_dir = uu.select_folder("大元のフォルダを選択してください")
    if base_dir is None:
        return

    print("-" * 30)

    # 日付と時間を無視するための正規表現
    #   ^(.*)    : 大元の名前 (例: "MPS5 settei")  -> group(1)
    #   _(\d{8}) : 8桁の日付 (例: "_20260618")
    #   _(\d{6}) : 6桁の時間 (例: "_121532")
    #   _(\d+)   : 連番 (例: "_000001")           -> group(4)
    #   (.*)$    : 拡張子など (例: ".csv")
    pattern = re.compile(r"^(.*)_(\d{8})_(\d{6})_(\d+)(.*)$")

    trashed_total = 0

    with uu.UndoJournal(base_dir, STEP_NAME) as j:
        for dirpath, dirnames, filenames in uu.walk(base_dir):
            target_groups = {}

            for filename in filenames:
                match = pattern.match(filename)
                if match:
                    prefix = match.group(1)        # "MPS5 settei" のような大元の名前
                    seq_num = int(match.group(4))  # "000001" のような連番
                    target_groups.setdefault(prefix, []).append((filename, seq_num))
                else:
                    print(f"【スキップ】ルール不一致: {filename}")

            # グループごとに退避判定
            for prefix, files in target_groups.items():
                if len(files) > KEEP_COUNT:
                    files.sort(key=lambda x: x[1])
                    trash_count = len(files) - KEEP_COUNT

                    for i in range(trash_count):
                        file_path = dirpath / files[i][0]
                        try:
                            j.trash(file_path)
                            print(f"退避しました: {file_path}")
                            trashed_total += 1
                        except Exception as e:
                            print(f"退避エラー ({file_path}): {e}")

    print("-" * 30)
    print(f"処理が完了しました。合計 {trashed_total} 個のファイルを _trash へ退避しました。")


main()

選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/202603-4 - 本実験圧力データ/202606・08-実験データ再々/202606・08-間接式樹脂圧力本実験データ再々/csv
------------------------------
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-間接式樹脂圧力本実験データ再々\csv\0.5mm‗futaba\190℃\020\MPS5 settei_20260618_121532_000001.csv
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-間接式樹脂圧力本実験データ再々\csv\0.5mm‗futaba\190℃\020\MPS5 settei_20260618_121600_000002.csv
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-間接式樹脂圧力本実験データ再々\csv\0.5mm‗futaba\190℃\020\MPS5 settei_20260618_121628_000003.csv
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-間接式樹脂圧力本実験データ再々\csv\0.5mm‗futaba\190℃\020\MPS5 settei_20260618_121656_000004.csv
退避しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-間接式樹脂圧力本実験データ再々\csv\0.5mm‗fu

---
### ⏪ 復元（元に戻す）

このセルを実行すると、**このノートブックで行った直前の1工程**を巻き戻します。
（メインフォルダの `_undo/undo_log.json` に記録された履歴を使います）

繰り返し実行すれば、01〜06 のどの工程まででもさかのぼれます。
削除したファイルは `_trash` フォルダに退避されているので、これも一緒に元の場所へ戻ります。

In [ ]:
# ===== 共通の復元セル =====
# 直前に実行した1工程を巻き戻します。
# 続けて実行すれば、さらに1つ前の工程へとさかのぼれます。

uu.undo_interactive()